$${\color{yellow}{\text{Deep Learning Webinar-1, Saturday, April 19, 2025}}}$$



---

Load essential libraries

---

In [ ]:
!pip install --upgrade numpy
!pip install --upgrade gensim

---

After reinstalling numpy and gensim, go to *Runtime -> Restart Session* and execute from the following cell onward:

---

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
plt.style.use('dark_background')
%matplotlib inline
import sys
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
import nltk
import gensim.downloader
from nltk.tokenize import word_tokenize

---

Mount Google Drive folder if running Google Colab

---

In [ ]:
## Mount Google drive folder if running in Colab
if('google.colab' in sys.modules):
    from google.colab import drive
    drive.mount('/content/drive', force_remount = True)
    DIR = '/content/drive/MyDrive/Colab Notebooks/MAHE/Office of Online Education/MDS6304_Webinar_April2025'
    DATA_DIR = DIR+'/Data/'
else:
    DATA_DIR = 'Data/'

Mounted at /content/drive


---

**We will now use Pytorch to create tensors**

The patient data matrix:

![patient data matrix](https://1drv.ms/i/s!AjTcbXuSD3I3hsxIkL4V93-CGq8RkQ?embed=1&width=1000)

**Notation**:

Zeroth patient vector $\mathbf{x}^{(0)}= \begin{bmatrix}72\\120\\37.3\\104\\32.5\end{bmatrix}$ and zeroth feature (heart rate vector) $\mathbf{x}_0 = \begin{bmatrix}72\\85\\68\\90\\84\\78\end{bmatrix}.$

---



In [ ]:
## Create a patient data matrix as a constant tensor
X = torch.tensor([[72, 120, 37.3, 104, 32.5],
                 [85, 130, 37.0, 110, 14],
                 [68, 110, 38.5, 125, 34],
                 [90, 140, 38.0, 130, 26],
                 [84, 132, 38.3, 146, 30],
                 [78, 128, 37.2, 102, 12]])
print(type(X))
print(X)
print(X.shape)
# X is an rank-2 tensor which is similar to a numpy 2D array
print(X[0]) # this is patient-0 info which is a rank-1 tensor
print(X[0, 2]) # the 2nd feature of the 0th patient (temperature of the 0th patient)
print(X.dtype)

<class 'torch.Tensor'>
tensor([[ 72.0000, 120.0000,  37.3000, 104.0000,  32.5000],
        [ 85.0000, 130.0000,  37.0000, 110.0000,  14.0000],
        [ 68.0000, 110.0000,  38.5000, 125.0000,  34.0000],
        [ 90.0000, 140.0000,  38.0000, 130.0000,  26.0000],
        [ 84.0000, 132.0000,  38.3000, 146.0000,  30.0000],
        [ 78.0000, 128.0000,  37.2000, 102.0000,  12.0000]])
torch.Size([6, 5])
tensor([ 72.0000, 120.0000,  37.3000, 104.0000,  32.5000])
tensor(37.3000)
torch.float32


---

**Convert a PyTorch object into a numpy array**

---

In [ ]:
X_numpy = X.numpy()
print(type(X_numpy))
print(X_numpy)
print(X_numpy.shape)

<class 'numpy.ndarray'>
[[ 72.  120.   37.3 104.   32.5]
 [ 85.  130.   37.  110.   14. ]
 [ 68.  110.   38.5 125.   34. ]
 [ 90.  140.   38.  130.   26. ]
 [ 84.  132.   38.3 146.   30. ]
 [ 78.  128.   37.2 102.   12. ]]
(6, 5)


---

**Addition and subtraction of vectors, scalar multiplication (apply operation componentwise)**

![vector addition](https://1drv.ms/i/c/37720f927b6ddc34/IQQ03G17kg9yIIA3NokBAAAAAZLAaAoWwhtn8Vk26NotALo?width=256)

![vector subtracton](https://1drv.ms/i/c/37720f927b6ddc34/IQQ03G17kg9yIIA3M4kBAAAAAU_n_mAEv006QFZm_sUj2Dc?width=256)

![vector multiplication](https://1drv.ms/i/c/37720f927b6ddc34/IQQ03G17kg9yIIA3NIkBAAAAAa_qL04bLT4kWoNeHcrR9LQ?width=256)

![vector geometry1](https://1drv.ms/i/c/37720f927b6ddc34/IQSGNMr5z3SSRry7LSKL7LybAcGYuzgw5smabV8-6DudXIs?width=230)

![vector geometry2](https://1drv.ms/i/c/37720f927b6ddc34/IQQ03G17kg9yIIA3WokBAAAAAQi8FPV9YCebl5WnyEKJ3vg?width=213&height=192)


---

In [ ]:
# Vector addition
print(X[1, :] + X[2, :])

# Vector subtraction
print(X[1, :] - X[2, :])

# Scalar-vector multiplication
print(X[:, 2]) # the temperature of all patients
# Operation not defined in pen & paper but in computation is referred to as
# broadcasting
print((9/5)*X[:, 2] + 32)

# Average patient
print((1/6) * (X[0, :] + X[1, :] + X[2, :] + X[3, :] + X[4, :] + X[5, :]))
print(torch.mean(X, dim = 0)) # dim = 0 means top-to-bottom

tensor([153.0000, 240.0000,  75.5000, 235.0000,  48.0000])
tensor([ 17.0000,  20.0000,  -1.5000, -15.0000, -20.0000])
tensor([37.3000, 37.0000, 38.5000, 38.0000, 38.3000, 37.2000])
tensor([ 99.1400,  98.6000, 101.3000, 100.4000, 100.9400,  98.9600])
tensor([ 79.5000, 126.6667,  37.7167, 119.5000,  24.7500])
tensor([ 79.5000, 126.6667,  37.7167, 119.5000,  24.7500])


---

Application of vector subtraction in natural language processing (NLP): download the word embedding model trained on Wikipedia articles.

---

In [ ]:
model = gensim.downloader.load('glove-wiki-gigaword-50')

[==================================================] 100.0% 66.0/66.0MB downloaded


---

Now we will see what embedding vector comes as a result of applying the model for the words *cricket* and *football*.

Next, we will do an *intuitive* subtraction of word embeddings as in

1. Cricket without Tendulkar
2. Football without Messi

Note that the embedding vectors have 50 components corresponding to the 50-dimensional embedding of model suggested by the name '**glove-wiki-gigaword-50**'

---

In [ ]:
print(model['cricket'])
print(model['football'])
a = model['cricket'] - model['tendulkar'] # cricket without tendulkar
b = model['football'] - model['messi'] # football without messi
print(a)
print(b)

[-1.2233    -0.28292   -1.7429     0.56621   -0.13787   -0.88088
 -0.26887    0.41893    0.94977   -0.61332    0.0092522  1.0045
 -0.89004   -0.55148    0.61202    0.42256    0.92935    0.83307
 -1.5568     0.084345  -0.018224   0.84803    0.68321    0.97059
  0.26561   -1.0538     0.40724   -0.45079   -0.89013    0.94153
  2.2055     0.75363    0.5166     0.47977    0.86824    0.57228
  0.81846   -0.070934  -0.9391    -0.81628   -0.35486   -0.010408
 -0.83316    1.1001    -0.087408   1.8452    -0.83112    0.43732
  0.63007   -0.81023  ]
[-1.8209    0.70094  -1.1403    0.34363  -0.42266  -0.92479  -1.3942
  0.28512  -0.78416  -0.52579   0.89627   0.35899  -0.80087  -0.34636
  1.0854   -0.087046  0.63411   1.1429   -1.6264    0.41326  -1.1283
 -0.16645   0.17424   0.99585  -0.81838  -1.7724    0.078281  0.13382
 -0.59779  -0.45068   2.5474    1.0693   -0.27017  -0.75646   0.24757
  1.0261    0.11329   0.17668  -0.23257  -1.1561   -0.10665  -0.25377
 -0.65102   0.32393  -0.58262   0.8813